# Rodada 4 — exclusões sistemáticas, sem escolher observações

Cada município e ano é excluído exatamente uma vez em cada destino. Escalas originais mantidas. Todos os refits são diagnósticos dependentes, não novos modelos selecionados pelo resultado.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d

## 1. Referência ampla

In [2]:
s=d.dropna(subset=['pp_agricultura','pp_pastagem','expo_soja','expo_milho']).copy()
for u in ['agricultura','pastagem']:e.linear('I00_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='diagnostico_influencia')

## 2. Retirar um município por vez
## 3. Retirar um ano por vez

In [3]:
retiradas=[]
for eixo in ['id','ano']:
    unidades=sorted(s[eixo].unique())
    for k,unidade in enumerate(unidades,1):
        menor=s[s[eixo].ne(unidade)]
        for u in ['agricultura','pastagem']:
            ident=f'I_{eixo}_{unidade}_{u}'
            e.linear(ident,'pp_'+u,['expo_soja','expo_milho'],menor,familia='diagnostico_influencia',detalhe=False)
            retiradas.append({'id':ident,'eixo_excluido':eixo,'unidade_excluida':str(unidade),'destino':u})
        if k%30==0:print(eixo,k,'unidades concluídas')
for row in e.joint:row['familia']='diagnostico_influencia'
e.salvar('influencia')
pd.DataFrame(retiradas).to_csv(e.p/'auditoria_retiradas.csv',index=False)
assert len(retiradas)==2*(s.id.nunique()+s.ano.nunique())
c=pd.DataFrame(e.coef);ref=c[c.id.str.startswith('I00')].copy();v=c[~c.id.str.startswith('I00')].merge(pd.DataFrame(retiradas),on='id')
rows=[]
for (eixo,u,termo),g in v.groupby(['eixo_excluido','destino','termo']):
    base=ref[(ref.id.eq('I00_'+u))&(ref.termo.eq(termo))].coef.iloc[0];j=(g.coef-base).abs().idxmax()
    rows.append({'eixo':eixo,'destino':u,'termo':termo,'referencia':base,'min_coef':g.coef.min(),'max_coef':g.coef.max(),'sinais_contrarios':int((g.coef*base<0).sum()),'maior_deslocamento_abs':abs(g.loc[j,'coef']-base),'unidade_maior_deslocamento':g.loc[j,'unidade_excluida'],'refits':len(g)})
a=pd.DataFrame(rows);a.to_csv(e.p/'RESUMO_INFLUENCIA.csv',index=False);display(a)

id 30 unidades concluídas


id 60 unidades concluídas


id 90 unidades concluídas


id 120 unidades concluídas


,eixo,destino,termo,referencia,min_coef,max_coef,sinais_contrarios,maior_deslocamento_abs,unidade_maior_deslocamento,refits
0,ano,agricultura,expo_milho,0.003751,-0.000835,0.007565,1,0.004586,2004,24
1,ano,agricultura,expo_soja,0.028791,0.012369,0.032468,0,0.016422,2003,24
2,ano,pastagem,expo_milho,0.058373,0.044362,0.075214,0,0.016841,2012,24
3,ano,pastagem,expo_soja,0.103806,0.091169,0.120156,0,0.016350,2022,24
4,id,agricultura,expo_milho,0.003751,-0.003246,0.006566,1,0.006997,5102678,137
5,id,agricultura,expo_soja,0.028791,0.020161,0.035639,0,0.008631,5104526,137
6,id,pastagem,expo_milho,0.058373,0.039181,0.082380,0,0.024007,5106505,137
7,id,pastagem,expo_soja,0.103806,0.093185,0.116092,0,0.012286,5100805,137
